<a href="https://colab.research.google.com/github/mahamtaqi3-cloud/Mapping-ALS-FTD-Relevant-Missense-Variants-on-the-C9orf72-SMCR8-WDR41-Arf-GAP-Complex/blob/main/Mapping_ALS_FTD_Relevant_Missense_Variants_on_the_C9orf72%E2%80%93SMCR8%E2%80%93WDR41_Arf_GAP_Complex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Setup

In [1]:
!pip -q install biopython py3Dmol

In [2]:
import io, os, re, copy, requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

GENES = {"C9orf72": "Q96LT7", "SMCR8": "Q8TEV9", "WDR41": "Q9HAD4"}   # human UniProt accessions

SEQ = {}
for gene, acc in GENES.items():
    j = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=60).json()
    name = j["genes"][0]["geneName"]["value"]
    SEQ[gene] = j["sequence"]["value"]
    print(f"{gene:8s} {acc}  UniProt gene name: {name:8s} length: {len(SEQ[gene])} aa")
    assert name.lower() == gene.lower(), f"{acc} does not look like {gene}; check the accession"

C9orf72  Q96LT7  UniProt gene name: C9orf72  length: 481 aa
SMCR8    Q8TEV9  UniProt gene name: SMCR8    length: 937 aa
WDR41    Q9HAD4  UniProt gene name: WDR41    length: 459 aa


Step 2: Find structures containing all three proteins

In [3]:
def rcsb_entries_with(acc):
    attr = "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers"
    query = {"query": {"type": "group", "logical_operator": "and", "nodes": [
        {"type": "terminal", "service": "text", "parameters": {"attribute": attr + ".database_accession", "operator": "exact_match", "value": acc}},
        {"type": "terminal", "service": "text", "parameters": {"attribute": attr + ".database_name", "operator": "exact_match", "value": "UniProt"}}]},
        "return_type": "entry", "request_options": {"return_all_hits": True}}
    r = requests.post("https://search.rcsb.org/rcsbsearch/v2/query", json=query, timeout=60)
    if r.status_code == 204:       # no hits
        return set()
    r.raise_for_status()
    return {(h["identifier"] if isinstance(h, dict) else h) for h in r.json().get("result_set", [])}

hits = {g: rcsb_entries_with(a) for g, a in GENES.items()}
for g, s in hits.items():
    print(g, "->", len(s), "PDB entries")
common = sorted(set.intersection(*hits.values()))
print("Entries containing all three:", len(common))

rows = []
for pid in common:
    j = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{pid}", timeout=60).json()
    info = j.get("rcsb_entry_info", {})
    rows.append(dict(pdb_id=pid, title=j["struct"]["title"], method=j["exptl"][0]["method"],
                     resolution=(info.get("resolution_combined") or [None])[0],
                     polymer_entities=info.get("polymer_entity_count"),
                     released=j["rcsb_accession_info"]["initial_release_date"][:10]))
if not rows:
    raise SystemExit("No entry contains all three proteins. Try only C9orf72 + SMCR8.")
structs = pd.DataFrame(rows).sort_values("resolution")
pd.set_option("display.max_colwidth", 100)
structs

C9orf72 -> 4 PDB entries
SMCR8 -> 6 PDB entries
WDR41 -> 4 PDB entries
Entries containing all three: 3


,pdb_id,title,method,resolution,polymer_entities,released
0,6LT0,cryo-EM structure of C9ORF72-SMCR8-WDR41,ELECTRON MICROSCOPY,3.20,3,2020-04-15
1,6V4U,Cryo-EM structure of SMCR8-C9orf72-WDR41 complex,ELECTRON MICROSCOPY,3.80,3,2020-09-02
2,7MGE,Structure of C9orf72:SMCR8:WDR41 in complex with ARF1,ELECTRON MICROSCOPY,3.94,4,2021-06-23


Step 3: Choose a structure and map chains to proteins

In [4]:
PDB_ID = None        # e.g. "1ABC" (fill in from the table). None = best resolution
if PDB_ID is None:
    PDB_ID = structs.iloc[0]["pdb_id"]
PDB_ID = PDB_ID.upper()
print("Using", PDB_ID)

os.makedirs("data", exist_ok=True)
cif_path = f"data/{PDB_ID}.cif"
r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.cif", timeout=180); r.raise_for_status()
open(cif_path, "w").write(r.text)

sifts = requests.get(f"https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{PDB_ID.lower()}", timeout=60).json()[PDB_ID.lower()]["UniProt"]
seg_rows = []
for gene, acc in GENES.items():
    for m in sifts.get(acc, {}).get("mappings", []):
        seg_rows.append(dict(gene=gene, chain=m["chain_id"], unp_start=m["unp_start"], unp_end=m["unp_end"],
                             pdb_start=m["start"]["residue_number"], pdb_end=m["end"]["residue_number"]))
segments = pd.DataFrame(seg_rows)
print(segments.to_string(index=False))

focus = segments.groupby("gene")["chain"].first().to_dict()    # one chain per protein used for annotation
chain_gene = dict(zip(segments.chain, segments.gene))            # every mapped chain -> protein
print("\nChains used for annotation:", focus)
print("Mapped chains (extra copies suggest a dimer of heterotrimers):", chain_gene)

Using 6LT0
   gene chain  unp_start  unp_end  pdb_start  pdb_end
C9orf72     C          1      481          1      481
C9orf72     F          1      481          1      481
  SMCR8     B          1      937          1      937
  SMCR8     E          1      937          1      937
  WDR41     A          1      459          1      459
  WDR41     D          1      459          1      459

Chains used for annotation: {'C9orf72': 'C', 'SMCR8': 'B', 'WDR41': 'A'}
Mapped chains (extra copies suggest a dimer of heterotrimers): {'C': 'C9orf72', 'F': 'C9orf72', 'B': 'SMCR8', 'E': 'SMCR8', 'A': 'WDR41', 'D': 'WDR41'}


Step 4: Collect missense variants

In [5]:
def uniprot_missense(gene, acc):
    r = requests.get(f"https://www.ebi.ac.uk/proteins/api/variation/{acc}",
                     headers={"Accept": "application/json"}, timeout=180)
    r.raise_for_status()
    out = []
    for f in r.json().get("features", []):
        wt, alt = f.get("wildType"), f.get("alternativeSequence")
        if not wt or not alt or len(wt) != 1 or len(alt) != 1 or alt == "*" or wt == alt:
            continue                                   # keep single-residue missense only
        try:
            pos = int(f["begin"])
        except (KeyError, ValueError, TypeError):
            continue
        freqs = []
        for p in f.get("populationFrequencies", []):
            try:
                freqs.append(float(p["frequency"]))
            except (KeyError, ValueError, TypeError):
                pass
        clin = sorted({c.get("type", "") for c in f.get("clinicalSignificances", []) if c.get("type")})
        out.append(dict(gene=gene, uniprot=acc, residue=pos, ref_aa=wt, alt_aa=alt,
                        protein_change=f"{wt}{pos}{alt}", clinical_significance="; ".join(clin),
                        max_pop_freq=max(freqs) if freqs else np.nan, source=f.get("sourceType", "")))
    return out

frames = []
for gene, acc in GENES.items():
    got = uniprot_missense(gene, acc)
    print(f"{gene}: {len(got)} missense variants")
    frames += got
variants = pd.DataFrame(frames)

# ---- OPTIONAL: add your own variants (ClinVar, literature) ----
# Upload a CSV (left sidebar -> Files) with columns: gene,protein_change   e.g. SMCR8,R145C or SMCR8,p.Arg145Cys
EXTRA_CSV = None     # e.g. "extra_variants.csv"
THREE = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
         "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"}
def one(code_):
    return code_.upper() if len(code_) == 1 else THREE.get(code_.capitalize())
if EXTRA_CSV:
    extra = []
    for _, row in pd.read_csv(EXTRA_CSV).iterrows():
        m = re.match(r"^(?:p\.)?\(?([A-Za-z]{1,3})(\d+)([A-Za-z]{1,3})\)?$", str(row.protein_change).strip())
        if not m or row.gene not in GENES or one(m.group(1)) is None or one(m.group(3)) is None:
            print("Could not use:", row.gene, row.protein_change); continue
        wt, pos, alt = one(m.group(1)), int(m.group(2)), one(m.group(3))
        extra.append(dict(gene=row.gene, uniprot=GENES[row.gene], residue=pos, ref_aa=wt, alt_aa=alt,
                          protein_change=f"{wt}{pos}{alt}", clinical_significance="", max_pop_freq=np.nan, source="manual"))
    variants = pd.concat([variants, pd.DataFrame(extra)], ignore_index=True)

# ---- sanity check: reference residue must match the UniProt sequence ----
ok = [(0 < r <= len(SEQ[g])) and SEQ[g][r - 1] == a for g, r, a in zip(variants.gene, variants.residue, variants.ref_aa)]
print("Reference residue mismatches (dropped):", int(len(ok) - sum(ok)))
variants = variants[ok].drop_duplicates(["gene", "residue", "ref_aa", "alt_aa"]).reset_index(drop=True)
print("Total unique missense variants:", len(variants))
variants.groupby("gene").size()

C9orf72: 560 missense variants
SMCR8: 1143 missense variants
WDR41: 532 missense variants
Reference residue mismatches (dropped): 194
Total unique missense variants: 2036


,0
gene,
C9orf72,537
SMCR8,1025
WDR41,474


Step 5: AlphaMissense scores

In [6]:
def alphamissense(acc):
    r = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{acc}", timeout=60); r.raise_for_status()
    url = r.json()[0].get("amAnnotationsUrl")
    if not url:
        return None
    return pd.read_csv(io.StringIO(requests.get(url, timeout=600).text))

am_frames = []
for gene, acc in GENES.items():
    d = alphamissense(acc)
    if d is None:
        print("No AlphaMissense file found for", gene); continue
    d = d.rename(columns={"protein_variant": "protein_change", "am_pathogenicity": "alphamissense"})
    d["gene"] = gene
    am_frames.append(d[["gene", "protein_change", "alphamissense", "am_class"]])
if am_frames:
    variants = variants.merge(pd.concat(am_frames), on=["gene", "protein_change"], how="left")
else:
    variants["alphamissense"], variants["am_class"] = np.nan, ""
print("Variants with an AlphaMissense score:", int(variants.alphamissense.notna().sum()), "of", len(variants))

Variants with an AlphaMissense score: 2036 of 2036


Step 6: Annotate variants on the structure

In [7]:
from Bio.PDB import MMCIFParser, ShrakeRupley, NeighborSearch
from Bio.PDB.Model import Model
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1

MAX_SASA = dict(A=129, R=274, N=195, D=193, C=167, E=223, Q=225, G=104, H=224, I=197,
                L=201, K=236, M=224, F=240, P=159, S=155, T=172, W=285, Y=263, V=174)   # Tien et al. 2013
INTERFACE_DELTA = 10.0   # A^2 buried on complex formation
CORE_RSA = 0.20
CONTACT = 4.5            # A, for naming partner chains

model = MMCIFParser(QUIET=True).get_structure(PDB_ID, cif_path)[0]
aa_residues = {(c.id, r.id[1]): r for c in model for r in c if is_aa(r, standard=True)}
resname1 = {k: protein_letters_3to1.get(r.get_resname().capitalize(), "X") for k, r in aa_residues.items()}

sr = ShrakeRupley()
sr.compute(model, level="R")
sasa_complex = {k: r.sasa for k, r in aa_residues.items()}

sasa_alone = {}
for c in model:
    m = Model(0); m.add(c.copy())
    sr.compute(m, level="R")
    for r in m[c.id]:
        if is_aa(r, standard=True):
            sasa_alone[(c.id, r.id[1])] = r.sasa

# names of every chain (to identify partners such as a bound GTPase)
d = MMCIF2Dict(cif_path)
aslist = lambda x: x if isinstance(x, list) else [x]
ent_desc = dict(zip(aslist(d["_entity.id"]), aslist(d["_entity.pdbx_description"])))
strand2ent = dict(zip(aslist(d["_pdbx_poly_seq_scheme.pdb_strand_id"]), aslist(d["_pdbx_poly_seq_scheme.entity_id"])))
chain_desc = {c.id: ent_desc.get(strand2ent.get(c.id), "?") for c in model}
print("Chains in this structure:")
for k, v in chain_desc.items():
    print(" ", k, "-", v)

ns = NeighborSearch([a for r in aa_residues.values() for a in r])
def partner_chains(key):
    own, found = key[0], set()
    for atom in aa_residues[key]:
        for ch in ns.search(atom.coord, CONTACT, level="C"):
            if ch.id != own:
                found.add(ch.id)
    return found

def label_partner(own_gene, ch):
    g = chain_gene.get(ch)
    if g is None: return "other"
    return "dimer" if g == own_gene else g

def to_pdb_resnum(gene, pos):
    seg = segments[(segments.gene == gene) & (segments.chain == focus[gene]) &
                   (segments.unp_start <= pos) & (segments.unp_end >= pos)]
    return None if seg.empty else int(seg.iloc[0].pdb_start + (pos - seg.iloc[0].unp_start))

def annotate(row):
    gene = row.gene
    blank = dict(chain="", pdb_resnum=np.nan, location_class="not_in_structure", rsa_alone=np.nan, partners="")
    if gene not in focus:
        return pd.Series(blank)
    num = to_pdb_resnum(gene, row.residue)
    key = (focus[gene], num)
    if num is None or key not in aa_residues:
        return pd.Series({**blank, "chain": focus[gene], "pdb_resnum": num})
    if resname1[key] != row.ref_aa:
        return pd.Series({**blank, "chain": focus[gene], "pdb_resnum": num, "location_class": "check_numbering"})
    rsa = sasa_alone[key] / MAX_SASA[row.ref_aa]
    if sasa_alone[key] - sasa_complex[key] > INTERFACE_DELTA:
        ps = partner_chains(key)
        labels = sorted({label_partner(gene, p) for p in ps}) or ["unspecified"]
        cls = "interface_" + "+".join(labels)
        desc = "; ".join(f"{p}:{chain_desc.get(p, '?')}" for p in sorted(ps))
    else:
        cls, desc = ("core" if rsa < CORE_RSA else "surface"), ""
    return pd.Series(dict(chain=focus[gene], pdb_resnum=num, location_class=cls, rsa_alone=round(rsa, 2), partners=desc))

old = [c for c in ["chain", "pdb_resnum", "location_class", "rsa_alone", "partners"] if c in variants]
variants = pd.concat([variants.drop(columns=old), variants.apply(annotate, axis=1)], axis=1)
print(variants.groupby(["gene", "location_class"]).size().unstack(fill_value=0))
print("\nNumbering check failures:", int((variants.location_class == "check_numbering").sum()))

Chains in this structure:
  A - WD repeat-containing protein 41
  B - Guanine nucleotide exchange protein SMCR8
  C - Guanine nucleotide exchange C9orf72
  D - WD repeat-containing protein 41
  E - Guanine nucleotide exchange protein SMCR8
  F - Guanine nucleotide exchange C9orf72
location_class  core  interface_C9orf72  interface_C9orf72+SMCR8  \
gene                                                               
C9orf72          184                  0                        0   
SMCR8            125                 55                        0   
WDR41            166                  2                        4   

location_class  interface_C9orf72+WDR41  interface_SMCR8  \
gene                                                       
C9orf72                               0               50   
SMCR8                                 4                0   
WDR41                                 0               29   

location_class  interface_SMCR8+WDR41  interface_SMCR8+dimer  interface_WDR4

Step 8: View in 3D and export a ChimeraX script

In [9]:
import py3Dmol

# Safety check: Step 6 must have run first
if "variants" not in globals() or "location_class" not in variants.columns:
    raise SystemExit("Run Step 6 first: 'variants' needs a 'location_class' column.")

def colour(cls):
    if cls.startswith("interface"):
        return "magenta" if "dimer" in cls else "orange"
    return {"core": "dodgerblue", "surface": "lightgreen"}.get(cls, "gray")

# Only variants that could be placed on the structure
plot_df = variants[variants.location_class.isin(["core", "surface"]) |
                   variants.location_class.str.startswith("interface")]
print("Variants to display:", len(plot_df))
print(plot_df.location_class.value_counts())

# Load the structure (PDB format if available, otherwise mmCIF)
resp = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb", timeout=120)
fmt, data = ("pdb", resp.text) if resp.ok else ("cif", open(cif_path).read())

# 3D view
view = py3Dmol.view(width=800, height=600)
view.addModel(data, fmt)
view.setStyle({"cartoon": {"color": "lightgray"}})
for _, r in plot_df.iterrows():
    view.addStyle({"chain": r.chain, "resi": int(r.pdb_resnum)},
                  {"stick": {"color": colour(r.location_class)}})
view.zoomTo()
view.show()

# ChimeraX script (same colours)
lines = [f"open {PDB_ID} format mmcif fromDatabase pdb", "cartoon", "color lightgray target c",
         "set bgColor white", "lighting soft"]
for _, r in plot_df.iterrows():
    sel = f"/{r.chain}:{int(r.pdb_resnum)}"
    lines += [f"show {sel} atoms", f"style {sel} stick", f"color {sel} {colour(r.location_class)}"]
open("map_variants.cxc", "w").write("\n".join(lines) + "\n")
print("Wrote map_variants.cxc")

Variants to display: 1182
location_class
surface                    501
core                       475
interface_SMCR8             79
interface_C9orf72           57
interface_WDR41             31
interface_unspecified       22
interface_SMCR8+WDR41        4
interface_SMCR8+dimer        4
interface_C9orf72+WDR41      4
interface_C9orf72+SMCR8      4
interface_dimer              1
Name: count, dtype: int64


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Wrote map_variants.cxc


Step 9: Save and download

In [10]:
# Safety check: Step 6 must have run first
if "variants" not in globals() or "location_class" not in variants.columns:
    raise SystemExit("Run Step 6 first: 'variants' needs a 'location_class' column.")

# Rebuild the candidate table if Step 7 was skipped
if "cands" not in globals() or "cols" not in globals():
    df = variants.copy()
    df["rare"] = df.max_pop_freq.isna() | (df.max_pop_freq < 1e-4)
    df["structural"] = df.location_class.str.startswith("interface") | (df.location_class == "core")
    df["am_high"] = df.alphamissense >= 0.564
    df["score"] = df[["rare", "structural", "am_high"]].sum(axis=1)
    cands = df.sort_values(["score", "alphamissense"], ascending=False)
    cols = ["gene", "protein_change", "location_class", "partners", "alphamissense",
            "max_pop_freq", "clinical_significance", "score"]

# Save results
variants.to_csv("csw_variants_annotated.csv", index=False)
cands[cols].head(20).to_csv("csw_top_candidates.csv", index=False)
print("Saved csw_variants_annotated.csv and csw_top_candidates.csv")

# Download whichever files exist
try:
    from google.colab import files
    for f in ["csw_variants_annotated.csv", "csw_top_candidates.csv",
              "variants_by_location.png", "map_variants.cxc"]:
        if os.path.exists(f):
            files.download(f)
        else:
            print("Skipped (not created yet):", f)
except ImportError:
    print("Not in Colab; files are in the working directory.")

Saved csw_variants_annotated.csv and csw_top_candidates.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Skipped (not created yet): variants_by_location.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>